In [ ]:
from pydantic import BaseModel
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
class Evaluation(BaseModel):
    TEDS: float
    filename: str
    is_complex: bool
    pred_ncols: int
    pred_nrows: int
    table_id: int
    true_ncols: int
    true_nrows: int
    timing: float

class BenchmarkStats(BaseModel):
    benchmark_name: str
    provider_name: str
    TEDS_mean: float
    TEDS_median: float
    TEDS_std: float
    timings_mean: float
    timings_std: float
    num_evaluations: int
    evaluations: list[Evaluation]

In [ ]:
benchmark_name = "PubTabNet"
provider_names = ["cells2table", "TableFormer"]

In [ ]:
all_stats: list[BenchmarkStats] = []
for provider_name in provider_names:
    with open(f"{benchmark_name}_{provider_name}.json", "r") as file:
        stats_json = file.read()
    all_stats.append(BenchmarkStats.model_validate_json(stats_json))

In [ ]:
plt.figure(figsize=(8,5))

for stats in all_stats:
    teds = [e.TEDS for e in stats.evaluations]
    plt.hist(teds, bins=20, alpha=0.6, label=stats.provider_name)

plt.xlabel("TEDS Score")
plt.ylabel("Frequency")
plt.title("TEDS Distribution")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

for stats in all_stats:
    sizes = [e.true_nrows * e.true_ncols for e in stats.evaluations]
    timing = [e.timing for e in stats.evaluations]

    plt.scatter(sizes, timing, alpha=0.5, label=stats.provider_name)

plt.xlabel("Table Size (number of true cells)")
plt.ylabel("Time (s)")
plt.title("Timing vs Table Size")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
for stats in all_stats:
    simple = [e.TEDS for e in stats.evaluations if not e.is_complex]
    complex_ = [e.TEDS for e in stats.evaluations if e.is_complex]

    plt.boxplot([simple, complex_], tick_labels=["Simple", "Complex"])
    plt.ylabel("TEDS")
    plt.title(f"Impact of Table Complexity on {stats.provider_name}")
    plt.grid(True)
    plt.show()

In [ ]:
for stats in all_stats:
    row_error = [abs(e.pred_nrows - e.true_nrows) for e in stats.evaluations]
    col_error = [abs(e.pred_ncols - e.true_ncols) for e in stats.evaluations]

    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)
    plt.hist(row_error, bins=20)
    plt.title("Row Prediction Error")

    plt.subplot(1,2,2)
    plt.hist(col_error, bins=20)
    plt.title("Column Prediction Error")

    plt.tight_layout()
    plt.show()

In [ ]:
for stats in all_stats:
    sizes = [e.true_nrows * e.true_ncols for e in stats.evaluations]
    teds = [e.TEDS for e in stats.evaluations]

    plt.scatter(sizes, teds, alpha=0.5)
    plt.xlabel("Table Size (#cells)")
    plt.ylabel("TEDS")
    plt.title("TEDS vs Table Size")
    plt.grid(True)
    plt.show()

In [ ]:
plt.figure(figsize=(8,5))
t =[]
n = []
for stats in all_stats:
    t.append([e.timing for e in stats.evaluations])
    n.append(stats.provider_name)

plt.boxplot(t, tick_labels=n)
plt.ylabel("Time (s)")
plt.title("Timing Comparison")
plt.grid(True)
plt.show()

In [ ]:
te = []
ti = []
n = []
for stats in all_stats:
    teds = [e.TEDS for e in stats.evaluations]
    timing = [e.timing for e in stats.evaluations]
    te.append(np.mean(teds))
    ti.append(np.mean(timing))
    n.append(stats.provider_name)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.bar(n, te)
plt.title("Average TEDS")

plt.subplot(1,2,2)
plt.bar(n, ti)
plt.title("Average Timing")

plt.tight_layout()
plt.show()